### MedGuard AI — Step 3: FastAPI Backend
---
**Is notebook mein kya hoga:**
- FastAPI server setup
- `/predict/diabetes` — diabetes risk + SHAP
- `/predict/heart` — heart disease risk + SHAP
- `/predict/combined` — dono ka combined risk score
- `/health` — server status

> Same folder mein rakho: `diabetes_model_v2.keras`, `diabetes_preprocessor_v2.pkl`, `diabetes_threshold_v2.json`, `heart_model_v2.pkl`, `heart_threshold_v2.json`

In [1]:
# Cell 2 — main.py file likho
main_py = '''
import json, joblib, warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import shap
import tensorflow as tf

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

# ─────────────────────────────────────────
# App setup
# ─────────────────────────────────────────
app = FastAPI(
    title="MedGuard AI API",
    description="Multimodal Disease Screening with Explainable Risk Scores",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ─────────────────────────────────────────
# Load models at startup
# ─────────────────────────────────────────
print("Loading models...")

# Diabetes
diabetes_model       = tf.keras.models.load_model("diabetes_model_v2.keras")
diabetes_preprocessor = joblib.load("diabetes_preprocessor_v2.pkl")
with open("diabetes_threshold_v2.json") as f:
    D_THRESHOLD = json.load(f)["threshold"]

# Heart
heart_pipeline = joblib.load("heart_model_v2.pkl")
with open("heart_threshold_v2.json") as f:
    H_THRESHOLD = json.load(f)["threshold"]

# SHAP explainers
heart_rf       = heart_pipeline.named_steps["model"]
heart_explainer = shap.TreeExplainer(heart_rf)

print(f"Diabetes model loaded  | Threshold: {D_THRESHOLD}")
print(f"Heart model loaded     | Threshold: {H_THRESHOLD}")
print("All models ready!")

# ─────────────────────────────────────────
# Input schemas
# ─────────────────────────────────────────
class DiabetesInput(BaseModel):
    Pregnancies     : float
    Glucose         : float
    BloodPressure   : float
    SkinThickness   : float
    Insulin         : float
    BMI             : float
    DiabetesPedigreeFunction: float
    Age             : float

class HeartInput(BaseModel):
    HighBP          : float
    HighChol        : float
    CholCheck       : float
    BMI             : float
    Smoker          : float
    Stroke          : float
    Diabetes        : float
    PhysActivity    : float
    Fruits          : float
    Veggies         : float
    HvyAlcoholConsump: float
    AnyHealthcare   : float
    NoDocbcCost     : float
    GenHlth         : float
    MentHlth        : float
    PhysHlth        : float
    DiffWalk        : float
    Sex             : float
    Age             : float
    Education       : float
    Income          : float

class CombinedInput(BaseModel):
    diabetes : DiabetesInput
    heart    : HeartInput

# ─────────────────────────────────────────
# Helper functions
# ─────────────────────────────────────────
def to_python(obj):
    """Numpy types ko JSON-safe Python types mein convert karo"""
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

def get_risk_zone(score: float) -> str:
    if score >= 65: return "RED"
    if score >= 35: return "YELLOW"
    return "GREEN"

def prep_diabetes(data: DiabetesInput):
    """Diabetes input preprocess"""
    import pandas as pd, numpy as np
    d = data.dict()
    df = pd.DataFrame([d])

    # Replace zeros
    zero_cols = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
    df[zero_cols] = df[zero_cols].replace(0, np.nan)

    # Feature engineering
    df["BMI_Age"]         = df["BMI"] / (df["Age"] + 1)
    df["Glucose_Insulin"] = df["Glucose"] * df["Insulin"]
    df["BP_Age"]          = df["BloodPressure"] / (df["Age"] + 1)

    return diabetes_preprocessor.transform(df)

def prep_heart(data: HeartInput):
    """Heart input preprocess"""
    df = pd.DataFrame([data.dict()])
    return df

def get_shap_diabetes(X_prep, feature_names):
    """Diabetes SHAP — KernelExplainer (background = zeros)"""
    background = np.zeros((1, X_prep.shape[1]))
    explainer  = shap.KernelExplainer(
        lambda x: diabetes_model.predict(x, verbose=0).ravel(),
        background
    )
    shap_vals = explainer.shap_values(X_prep, nsamples=50)
    shap_arr  = np.array(shap_vals).flatten()[:len(feature_names)]
    top = sorted(zip(feature_names, shap_arr), key=lambda x: abs(x[1]), reverse=True)[:6]
    return [{'feature': f, 'shap_value': round(float(s), 4),
             'impact': 'increases_risk' if s > 0 else 'decreases_risk'} for f, s in top]

def get_shap_heart(X_df, feature_names):
    """Heart SHAP — TreeExplainer (fast)"""
    heart_preprocess = heart_pipeline.named_steps.get(
        "preprocess", heart_pipeline.named_steps.get("scaler")
    )
    X_prep    = heart_preprocess.transform(X_df)
    shap_vals = heart_explainer.shap_values(X_prep)

    # (1, 21, 2) → class 1
    if isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
        shap_arr = shap_vals[0, :, 1]
    elif isinstance(shap_vals, list):
        shap_arr = np.array(shap_vals[1]).flatten()
    else:
        shap_arr = np.array(shap_vals).flatten()

    top = sorted(zip(feature_names, shap_arr), key=lambda x: abs(x[1]), reverse=True)[:6]
    return [{'feature': f, 'shap_value': round(float(s), 4),
             'impact': 'increases_risk' if s > 0 else 'decreases_risk'} for f, s in top]

# ─────────────────────────────────────────
# Routes
# ─────────────────────────────────────────
@app.get("/")
def root():
    return {"message": "MedGuard AI API is running!", "version": "1.0.0"}

@app.get("/health")
def health():
    return {
        "status"  : "ok",
        "models"  : {"diabetes": "loaded", "heart": "loaded"},
        "thresholds": {"diabetes": D_THRESHOLD, "heart": H_THRESHOLD}
    }

@app.post("/predict/diabetes")
def predict_diabetes(data: DiabetesInput):
    try:
        feature_names = [
            "Pregnancies","Glucose","BloodPressure","SkinThickness",
            "Insulin","BMI","DiabetesPedigreeFunction","Age",
            "BMI_Age","Glucose_Insulin","BP_Age"
        ]
        X_prep = prep_diabetes(data)
        prob   = float(diabetes_model.predict(X_prep, verbose=0).ravel()[0])
        pred   = int(prob >= D_THRESHOLD)
        score  = round(prob * 100, 1)
        shap_factors = get_shap_diabetes(X_prep, feature_names)

        return {
            "model"       : "diabetes",
            "probability" : score,
            "prediction"  : "Diabetic" if pred else "Not Diabetic",
            "risk_zone"   : get_risk_zone(score),
            "threshold"   : D_THRESHOLD,
            "top_factors" : shap_factors
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict/heart")
def predict_heart(data: HeartInput):
    try:
        feature_names = list(data.dict().keys())
        X_df   = prep_heart(data)
        prob   = float(heart_pipeline.predict_proba(X_df)[0][1])
        pred   = int(prob >= H_THRESHOLD)
        score  = round(prob * 100, 1)
        shap_factors = get_shap_heart(X_df, feature_names)

        return {
            "model"       : "heart_disease",
            "probability" : score,
            "prediction"  : "Heart Disease" if pred else "No Disease",
            "risk_zone"   : get_risk_zone(score),
            "threshold"   : H_THRESHOLD,
            "top_factors" : shap_factors
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict/combined")
def predict_combined(data: CombinedInput):
    try:
        # Diabetes
        d_feature_names = [
            "Pregnancies","Glucose","BloodPressure","SkinThickness",
            "Insulin","BMI","DiabetesPedigreeFunction","Age",
            "BMI_Age","Glucose_Insulin","BP_Age"
        ]
        X_d    = prep_diabetes(data.diabetes)
        d_prob = float(diabetes_model.predict(X_d, verbose=0).ravel()[0])
        d_pred = int(d_prob >= D_THRESHOLD)
        d_shap = get_shap_diabetes(X_d, d_feature_names)

        # Heart
        h_feature_names = list(data.heart.dict().keys())
        X_h    = prep_heart(data.heart)
        h_prob = float(heart_pipeline.predict_proba(X_h)[0][1])
        h_pred = int(h_prob >= H_THRESHOLD)
        h_shap = get_shap_heart(X_h, h_feature_names)

        # Combined risk
        combined_score = round((d_prob * 0.5 + h_prob * 0.5) * 100, 1)
        zone = get_risk_zone(combined_score)

        return {
            "risk_score" : combined_score,
            "risk_zone"  : zone,
            "alert"      : zone == "RED",
            "diabetes": {
                "probability": round(d_prob * 100, 1),
                "prediction" : "Diabetic" if d_pred else "Not Diabetic",
                "risk_zone"  : get_risk_zone(round(d_prob*100,1)),
                "top_factors": d_shap
            },
            "heart_disease": {
                "probability": round(h_prob * 100, 1),
                "prediction" : "Heart Disease" if h_pred else "No Disease",
                "risk_zone"  : get_risk_zone(round(h_prob*100,1)),
                "top_factors": h_shap
            }
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

with open('main.py', 'w', encoding='utf-8') as f:
    f.write(main_py)

print('main.py created!')

main.py created!


In [ ]:
# Cell 3 — Server start karo
# Ye cell run karo — server start ho jayega
# Browser mein jao: http://localhost:8000/docs
import subprocess, sys

print('Server start ho raha hai...')
print('Browser mein jao: http://localhost:8000/docs')
print('Stop karne ke liye: Kernel > Interrupt')
print()

!uvicorn main:app --host 0.0.0.0 --port 8000 --reload

Server start ho raha hai...
Browser mein jao: http://localhost:8000/docs
Stop karne ke liye: Kernel > Interrupt

